In [1]:
# variables = ['datadate', 'ticker', 'agr', 'bm', 'mom12m', 'mve', 'operprof', 'roeq', 'absacc', 'acc', 'aeavol', 'age', 'baspread', 'BETA', 'bm_ia', 
#              'cash', 'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho', 'chempia', 'chfeps', 'chinv', 'chmom', 'chnanalyst', 
#              'chpmia', 'chtx', 'cinvest', 'convind', 'currat', 'depr', 'disp', 'divi', 'divo', 'dy', 'ear', 'egr', 'ep', 'fgr5yr', 
#              'gma', 'grcapx', 'grltnoa', 'herf', 'hire', 'idiovol', 'ill', 'indmom', 'invest', 'IPO', 'lev', 'mom1m', 'mom36m', 'ms', 
#              'mve_ia', 'nanalyst', 'nincr', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr', 'pchgm_pchsale', 'pchsale_pchinvt', 
#              'pchsale_pchrect', 'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'pricedelay', 'ps', 'rd', 'rd_mve', 'rd_sale', 'realestate', 
#              'retvol', 'roaq', 'roavol', 'roic', 'rsup', 'salecash', 'saleinv', 'salerec', 'secured', 'securedind', 'sfe', 'sgr', 'sin', 
#              'sp', 'std_dolvol', 'std_turn', 'stdcf', 'sue', 'tang', 'tb', 'turn', 'zerotrade']
variables = ['datadate', 'ticker', 'bm', 'mom12m', 'mve']

In [2]:
# import pandas as pd
# import numpy as np

# green_cleaned = pd.read_csv('../green cleaned.csv')
# green_cleaned['ret_fwd_1'] = (green_cleaned.groupby('permno')['ret_excess'].shift(-1) )

In [3]:
# green_cleaned = green_cleaned[green_cleaned['datadate'] < '2020-01-31']
# green_cleaned.loc[green_cleaned['datadate'] == '2019-12-31', 'ret_fwd_1'] = np.nan

In [4]:
# green_cleaned.to_csv('green_cleaned_pre_2020.csv', index=False)

In [5]:
from dotenv import find_dotenv, load_dotenv
from getpass import getpass
import os

load_dotenv(find_dotenv(), override=True)
#os.environ["OPENAI_MODEL_ID"] = "openai/gpt-4o-2024-08-06"
os.environ["OPENAI_MODEL_ID"] = "openai/gpt-3.5-turbo-0125"
os.environ.pop("GEMINI_API_KEY", None)

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY:")

In [41]:
import sqlite3
import pandas as pd
from crewai.tools import tool
from crewai import Agent, Task, Crew, Process
from agentics import AG
import time
from functools import wraps
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# ============================================
# Rate Limiting Setup
# ============================================
class RateLimiter:
    def __init__(self, calls_per_minute=5):
        self.calls_per_minute = calls_per_minute
        self.interval = 60.0 / calls_per_minute
        self.last_call = 0
        self.call_count = 0
        
    def wait_if_needed(self):
        """Wait if necessary to respect rate limit."""
        current_time = time.time()
        time_since_last = current_time - self.last_call
        
        if time_since_last < self.interval:
            sleep_time = self.interval - time_since_last
            print(f"[Rate Limit] Waiting {sleep_time:.2f}s (Call #{self.call_count + 1})")
            time.sleep(sleep_time)
        
        self.last_call = time.time()
        self.call_count += 1

rate_limiter = RateLimiter(calls_per_minute=5)

def rate_limit(func):
    """Decorator with retry logic for rate limiting."""
    @wraps(func)
    @retry(
        retry=retry_if_exception_type((Exception,)),
        wait=wait_exponential(multiplier=1, min=4, max=60),
        stop=stop_after_attempt(5)
    )
    def wrapper(*args, **kwargs):
        rate_limiter.wait_if_needed()
        try:
            return func(*args, **kwargs)
        except Exception as e:
            print(f"[Error] {func.__name__} failed: {str(e)}")
            raise
    return wrapper

# ============================================
# Database Setup
# ============================================
def setup_firm_database(csv_path, db_path='firm_data.db', selected_columns=None):
    """Load your CSV into SQLite database (run this once)."""
    print(f"Loading CSV from {csv_path}...")
    
    sample_df = pd.read_csv(csv_path, nrows=5)
    
    if selected_columns is None:
        print(f"\nAvailable columns ({len(sample_df.columns)} total):")
        for i, col in enumerate(sample_df.columns, 1):
            print(f"  {i}. {col}")
        
        response = input("\nPress Enter to use ALL columns, or type 'stop' to exit: ").strip()
        if response.lower() == 'stop':
            print("Exiting.")
            return
        selected_columns = sample_df.columns.tolist()
    else:
        missing = [col for col in selected_columns if col not in sample_df.columns]
        if missing:
            print(f"\n❌ Error: These columns don't exist: {missing}")
            return
    
    clean_columns = [col.replace(' ', '_').replace('-', '_').replace('.', '_')
                    .replace('(', '').replace(')', '').replace('/', '_') 
                    for col in selected_columns]
    
    import os
    if os.path.exists(db_path):
        os.remove(db_path)
    
    chunk_size = 50000
    conn = sqlite3.connect(db_path)
    
    first_chunk = True
    chunk_num = 0
    total_rows = 0
    
    print("\nLoading data...")
    for chunk in pd.read_csv(csv_path, chunksize=chunk_size, usecols=selected_columns):
        chunk.columns = clean_columns
        chunk_num += 1
        total_rows += len(chunk)
        
        if first_chunk:
            chunk.to_sql('firms', conn, if_exists='replace', index=False)
            first_chunk = False
        else:
            chunk.to_sql('firms', conn, if_exists='append', index=False)
        
        print(f"  Chunk {chunk_num}: {total_rows:,} rows")
    
    cursor = conn.cursor()
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_ticker ON firms(ticker)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_date ON firms(datadate)")
    conn.commit()
    conn.close()
    
    print(f"\n✓ Database created: {db_path} ({total_rows:,} rows)")
    return clean_columns

# ============================================
# Database Query Tools
# ============================================
@tool("query_firm_database")
@rate_limit
def query_firm_database(sql_query: str) -> str:
    """
    Execute SQL queries on the S&P 500 firm characteristics database.
    
    Available table: 'firms' (contains all firm characteristic data)
    
    IMPORTANT DATA NOTES:
    - 'mve' column = log(market value of equity), i.e., log size
    - ALL features are standardized: mean = 0, standard deviation = 1
    - Higher values = more extreme positive, lower values = more extreme negative
    
    Example queries:
    - SELECT * FROM firms WHERE datadate = '2020-12-31'
    - SELECT AVG(bm), AVG(mom12m), AVG(mve) FROM firms WHERE datadate = '2020-12-31'
    - SELECT ticker, bm, mom12m FROM firms WHERE datadate = '2020-12-31' AND bm < -1 ORDER BY bm LIMIT 20
    
    Args:
        sql_query: A valid SQL SELECT query
    
    Returns:
        Query results as a formatted string
    """
    try:
        conn = sqlite3.connect('../GPT 3.5/firm_data.db')
        df = pd.read_sql_query(sql_query, conn)
        conn.close()
        
        if df.empty:
            return "Query returned no results."
        
        limit = 10
        result = f"Query returned {len(df)} rows"
        if len(df) > limit:
            result += f" (showing first {limit} to fit context):\n\n"
            result += df.head(limit).to_string(index=False)
        else:
            result += ":\n\n" + df.to_string(index=False)
        
        return result
    except Exception as e:
        return f"Error executing query: {str(e)}"

@tool("get_database_schema")
@rate_limit
def get_database_schema() -> str:
    """
    Get the schema of the firms database including all column names and types.
    
    CRITICAL DATA INFORMATION:
    - 'mve' = log(market value of equity), represents log firm size
    - ALL features are standardized to mean=0, std=1
    - Values represent standard deviations from mean (z-scores)
    """
    try:
        conn = sqlite3.connect('../GPT 3.5/firm_data.db')
        cursor = conn.cursor()
        
        cursor.execute("PRAGMA table_info(firms)")
        columns = cursor.fetchall()
        
        cursor.execute("SELECT COUNT(*) FROM firms")
        row_count = cursor.fetchone()[0]
        
        cursor.execute("SELECT MIN(datadate), MAX(datadate) FROM firms")
        date_range = cursor.fetchone()
        
        conn.close()
        
        schema = f"""Database: ../GPT 3.5/firm_data.db
Table: firms
Total rows: {row_count:,}
Date range: {date_range[0]} to {date_range[1]}

DATA STANDARDIZATION:
- 'mve' column = log(market value of equity) - represents log firm size
- 'bm' column = log(book to market) - a high bm value means a firm is undervalued, and a low bm value means a firm is overvalued
- 'mom12m' column = 12 month momentum
- ALL features are standardized: mean = 0, standard deviation = 1
- Interpret values as z-scores (standard deviations from mean)
- Example: A value of 2.0 means 2 standard deviations above average
- Example: A value of -1.5 means 1.5 standard deviations below average

Columns ({len(columns)} total):
"""
        for col in columns:
            schema += f"  - {col[1]} ({col[2]})\n"
        
        return schema
    except Exception as e:
        return f"Error retrieving schema: {str(e)}"

@tool("get_extreme_firms")
@rate_limit
def get_extreme_firms(characteristic: str, threshold: float, date: str = None) -> str:
    """
    Find firms with extreme values for a given characteristic.
    
    Args:
        characteristic: Column name (e.g., 'bm', 'mom12m', 'mve')
        threshold: Z-score threshold (e.g., 1.5 for values > 1.5 or < -1.5)
        date: Date in format 'YYYY-MM-DD' (if None, uses latest date)
    
    Returns:
        Firms with extreme values
    """
    try:
        conn = sqlite3.connect('../GPT 3.5/firm_data.db')
        
        if date is None:
            cursor = conn.cursor()
            cursor.execute("SELECT MAX(datadate) FROM firms")
            date = cursor.fetchone()[0]
        
        query = f"""
        SELECT ticker, datadate, mve, bm, mom12m
        FROM firms
        WHERE datadate = '{date}' 
        AND (ABS({characteristic}) > {threshold})
        ORDER BY ABS({characteristic}) DESC
        """
        
        df = pd.read_sql_query(query, conn)
        conn.close()
        
        if df.empty:
            return f"No firms found with |{characteristic}| > {threshold} at {date}"
        
        limit = 10
        result = f"Firms with extreme {characteristic} (|z-score| > {threshold}) at {date} ({len(df)} found):\n\n"
        if len(df) > limit:
            result += df.head(limit).to_string(index=False) + f"\n... (and {len(df) - limit} more matching rows)"
        else:
            result += df.to_string(index=False)
        
        return result
    except Exception as e:
        return f"Error finding extreme firms: {str(e)}"


@tool("test_complex_condition")
@rate_limit
def test_complex_condition(sql_formula: str, date: str = None) -> str:
    """
    Test a continuous scoring function on the database to see which firms end up in the top and bottom deciles.
    
    Args:
        sql_formula: A valid SQL mathematical expression using mve, bm, and mom12m. 
                     DO NOT USE 'if/else' or 'WHERE' clauses. Do not use `**` for powers. Use `mve * mve` instead of `mve**2`.
        date: Date in format 'YYYY-MM-DD' (if None, uses latest date)
    
    Returns:
        Statistics about the score distribution and samples from the Top 10% (Buy) and Bottom 10% (Sell).
    """
    try:
        conn = sqlite3.connect('../GPT 3.5/firm_data.db')
        
        if date is None:
            cursor = conn.cursor()
            cursor.execute("SELECT MAX(datadate) FROM firms")
            date = cursor.fetchone()[0]
        
        # Calculate the score for all valid firms
        query = f"""
        SELECT ticker, mve, bm, mom12m, ({sql_formula}) as score
        FROM firms
        WHERE datadate = '{date}' 
        AND mve IS NOT NULL AND bm IS NOT NULL AND mom12m IS NOT NULL
        ORDER BY score DESC
        """
        
        df = pd.read_sql_query(query, conn)
        conn.close()
        
        if df.empty:
            return "No valid firms found to score."

        total_firms = len(df)
        decile_size = int(total_firms * 0.10)
        
        if decile_size == 0:
            return "Not enough firms to calculate deciles."

        top_decile = df.head(decile_size)
        bottom_decile = df.tail(decile_size)

        result = f"""Scoring Function Test at {date}:

Formula: {sql_formula}
Total firms scored: {total_firms}

=== TOP DECILE (BUY CANDIDATES) ===
Average Score: {top_decile['score'].mean():.2f}
Average Size (mve): {top_decile['mve'].mean():.2f} z-scores
Average Value (bm): {top_decile['bm'].mean():.2f} z-scores
Average Momentum (mom12m): {top_decile['mom12m'].mean():.2f} z-scores

Sample Top Firms:
{top_decile.head(5).to_string(index=False)}

=== BOTTOM DECILE (SELL CANDIDATES) ===
Average Score: {bottom_decile['score'].mean():.2f}
Average Size (mve): {bottom_decile['mve'].mean():.2f} z-scores
Average Value (bm): {bottom_decile['bm'].mean():.2f} z-scores
Average Momentum (mom12m): {bottom_decile['mom12m'].mean():.2f} z-scores

Sample Bottom Firms:
{bottom_decile.tail(5).to_string(index=False)}
"""
        return result
    except Exception as e:
        return f"Error evaluating formula: {str(e)}\nMake sure your input is a valid mathematical SQL expression without WHERE clauses."

# ============================================
# Configure LLM
# ============================================
def get_rate_limited_llm():
    """Get LLM provider with rate limiting configured."""
    llm = AG.get_llm_provider("openai")
    try:
        llm.max_retries = 5
        llm.timeout = 60
    except:
        pass
    return llm

# ============================================
# Strategy Development Agent (Single Agent)
# ============================================
strategy_agent = Agent(
    role="Quantitative Strategy Developer",
    goal="Develop systematic BUY/HOLD/SELL rules based on firm characteristics that can be applied to all S&P 500 firms",
    backstory="""
    You are an expert quantitative strategist who develops non-linear alpha models.
    
    CRITICAL DATA UNDERSTANDING:
    - 'mve' = log(market value of equity), represents log firm size
    - 'bm' = book-to-market ratio (value factor). High bm = undervalued, low bm = overvalued.
    - 'mom12m' = 12-month momentum
    - ALL features are standardized: mean = 0, standard deviation = 1
    
    Your task is to develop a NON-LINEAR PROBABILITY FUNCTION using these 3 factors. This function should output a score (or probability) indicating the attractiveness of buying a stock. The trading desk will then rank these scores, buying the top decile (top 10%) and selling the bottom decile (bottom 10%).
    
    1. EXPLORE THE DATA (December 2020):
       - Use causal masking from December 2020 to prevent look-ahead bias.
       - Consider how the 3 factors interact non-linearly (e.g., does momentum only matter for small cap stocks? Does value compound with momentum?)
    
    2. DEVELOP THE SCORING FUNCTION:
       - Do not give simple "if/then -> BUY/SELL" rules. 
       - Instead, design a mathematical formula or pseudo-code function that takes mve, bm, and mom12m and outputs a continuous score.
       - You can use non-linear transformations (e.g., squared terms, thresholds, sigmoid activations) or interaction terms (e.g., bm * mom12m).
    
    3. PROVIDE RATIONALE:
       - Explain the economic intuition behind your chosen non-linearities and factor interactions.
    
    OUTPUT FORMAT:
    ===========================================
    NON-LINEAR PROBABILITY MODEL
    ===========================================
    
    Data Exploration Summary:
    - [Key interactions and patterns observed]
    
    PROBABILITY FUNCTION (Python/Pseudo-code):
    def calculate_buy_probability(mve, bm, mom12m):
        # You MUST invent your own unique mathematical formula here based on your testing.
        # Include linear terms, interaction terms (like variable1 * variable2), 
        # or non-linearities (like variable**2 or variable**3) as you see fit.
        # DO NOT output a generic example. Determine specific, precise numerical weights yourself.
        score = ... 
        return score
    
    Rationale:
    - [Economic reasoning for the interactions and weights]

    FINAL RULE: You are not allowed to submit your response until you have successfully executed the `test_scoring_function` tool at least once and analyzed the Top 10% and Bottom 10% results.
    ===========================================
    """,
    llm=get_rate_limited_llm(),
    memory=True, # Note: We leave memory=True, but we've stopped tools from breaking it.
    allow_delegation=False,
    verbose=True,
    max_iter=15,
    max_rpm=10
)

strategy_task = Task(
    description="""
    Develop a non-linear probability scoring function for S&P 500 firms at December 2020.
    
    Available characteristics (all are z-scores):
    - mve: log firm size
    - bm: log book-to-market (value)
    - mom12m: 12-month momentum
    
    Your process:
    1. Explore the data for December 2020 using the provided tools.
    2. Look for non-linear relationships and interactions between the 3 factors.
    3. Develop a mathematical scoring function that outputs a continuous "buy probability" score.
    
    CRITICAL: Output an explicit mathematical formula or Python function. The trading desk will use your formula to rank all 500 stocks, buying the top 10% and selling the bottom 10%. Use causal masking to prevent look-ahead bias. 
    YOU MUST CREATE YOUR OWN UNIQUE WEIGHTS AND FORMULA. DO NOT copy any generic examples. Your specific mathematical choices must be driven by your internal financial logic and tool testing.
    """,
    expected_output="""
    A complete model document following the EXACT format requested.
    
    CRITICAL INSTRUCTIONS TO PREVENT EARLY COMPLETION:
    - DO NOT return your plan of action (e.g., "I will start by exploring...") as your final answer.
    - You MUST actually invoke the tools (get_database_schema, test_scoring_function) BEFORE generating your final response.
    - Your final response MUST contain the actual Python code and the detailed Economic Rationale.
    
    Required Sections:
    1. Data exploration summary.
    2. A Python function (calculate_buy_probability) using your UNIQUE formula for mve, bm, and mom12m.
    3. A Specific Economic Rationale section explaining the fundamental/behavioral logic for every term.
    """,
    agent=strategy_agent,
    tools=[
        get_database_schema,
        query_firm_database,
        get_extreme_firms,
        test_complex_condition
    ]
)

crew = Crew(
    agents=[strategy_agent],
    tasks=[strategy_task],
    process=Process.sequential,
    memory=False,
    verbose=True,
    max_rpm=10
)

# ============================================
# Main Execution
# ============================================
if __name__ == "__main__":

    # selected_cols = variables  # Add your columns
    # setup_firm_database('green cleaned.csv', selected_columns=selected_cols)
    
    print("=" * 70)
    print("SYSTEMATIC STRATEGY DEVELOPER")
    print("=" * 70)
    print("\nGenerates systematic BUY/HOLD/SELL rules for S&P 500 firms")
    print("Based on z-score analysis of firm characteristics")
    print("\nThe agent will:")
    print("  1. Explore your firm characteristics data")
    print("  2. Develop systematic rules with specific thresholds")
    print("  3. Output implementation-ready strategy")
    print("\nYou can then apply the rules to all firms manually.")
    print("=" * 70)
    
    input("\nPress Enter to start strategy development...")
    
    try:
        print("\n🚀 Starting strategy development...\n")
        result = crew.kickoff()
        
        print("\n" + "=" * 70)
        print("SYSTEMATIC STRATEGY OUTPUT")
        print("=" * 70)
        print(result)
        print("=" * 70)
        
        # Save to file
        with open('systematic_strategy_2021_gpt3.5.txt', 'w') as f:
            f.write(str(result))
        
        print("\n✓ Strategy saved to 'systematic_strategy_2021_gpt3.5.txt'")
        print("\nNext steps:")
        print("  1. Review the strategy rules")
        print("  2. Implement in Python/pandas using the thresholds")
        print("  3. Apply to all firms in your dataset")
        print("  4. Generate signals for backtesting")
        
    except KeyboardInterrupt:
        print("\n\n⏸ Interrupted.")
    except Exception as e:
        print(f"\n❌ Error: {e}")
        if "429" in str(e) or "RateLimitError" in str(e):
            print("\n🚨 Rate limit hit. Wait 60 seconds and try again.")


SYSTEMATIC STRATEGY DEVELOPER

Generates systematic BUY/HOLD/SELL rules for S&P 500 firms
Based on z-score analysis of firm characteristics

The agent will:
  1. Explore your firm characteristics data
  2. Develop systematic rules with specific thresholds
  3. Output implementation-ready strategy

You can then apply the rules to all firms manually.

🚀 Starting strategy development...



╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1d9c5885-3e56-4c57-be14-064b4e40da1a                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quantitative Strategy Developer                                                                         │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Develop a non-linear probability scoring function for S&P 500 firms at December 2020.                      │
│                                                                                                                 │
│      Available characteristics (all are z-scores):                                                              │
│      - mve: log firm size                                                                                       │
│      - bm: log book-to-market (value)                                                                           │
│      - mom12m: 12-month momentum                                                                                │
│                                                                                                                 │
│      Your process:                                                                                              │
│      1. Explore the data for December 2020 using the provided tools.                                            │
│      2. Look for non-linear relationships and interactions between the 3 factors.                               │
│      3. Develop a mathematical scoring function that outputs a continuous "buy probability" score.              │
│                                                                                                                 │
│      CRITICAL: Output an explicit mathematical formula or Python function. The trading desk will use your       │
│  formula to rank all 500 stocks, buying the top 10% and selling the bottom 10%. Use causal masking to prevent   │
│  look-ahead bias.                                                                                               │
│      YOU MUST CREATE YOUR OWN UNIQUE WEIGHTS AND FORMULA. DO NOT copy any generic examples. Your specific       │
│  mathematical choices must be driven by your internal financial logic and tool testing.                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quantitative Strategy Developer                                                                         │
│                                                                                                                 │
│  Thought: I will start by testing the scoring function using the `test_complex_condition` tool to understand    │
│  how my mathematical formula performs in practice. This will allow me to see which firms end up in the top and  │
│  bottom deciles based on the calculated buy probability score. This testing is crucial before finalizing the    │
│  scoring function.                                                                                              │
│                                                                                                                 │
│  Using Tool: test_complex_condition                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quantitative Strategy Developer                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```                                                                                                            │
│  ===========================================                                                                    │
│  NON-LINEAR PROBABILITY MODEL                                                                                   │
│  ===========================================                                                                    │
│                                                                                                                 │
│  Data Exploration Summary:                                                                                      │
│  - The scoring function formula "mve * bm + bm * mom12m + mom12m * mom12m" was tested on S&P 500 firms data     │
│  for December 2020, revealing significant differences in scores between the top and bottom deciles.             │
│                                                                                                                 │
│  PROBABILITY FUNCTION (Python/Pseudo-code):                                                                     │
│  def calculate_buy_probability(mve, bm, mom12m):                                                                │
│      # Non-linear scoring function                                                                              │
│      score = mve * bm + bm * mom12m + mom12m * mom12m                                                           │
│      return score                                                                                               │
│                                                                                                                 │
│  Rationale:                                                                                                     │
│  - The scoring function incorporates interactions between firm size (mve), book-to-market ratio (bm), and       │
│  12-month momentum (mom12m).                                                                                    │
│  - The term mve * bm captures the combined effect of firm size and value factor, indicating potentially         │
│  undervalued larger firms.                                                                                      │
│  - The term bm * mom12m considers the interaction between value and momentum, suggesting momentum effects on    │
│  undervalued stocks.                                                                                            │
│  - The term mom12m * mom12m introduces non-linearity by emphasizing the momentum effect squared, reflecting     │
│  potential momentum compounding.                                                                                │
│                                                                                                                 │
│  FINAL RULE: The trading desk will buy the top decile and sell the bottom decile of S&P 500 stocks based on     │
│  the calculated buy probability scores from the non-linear scoring function.                                    │
│  ===========================================                                                                    │
│  ```                                                                                                            │
│                                                        

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: cafda21c-7ac8-4d77-a11d-db48afb92130                                                                     │
│  Agent: Quantitative Strategy Developer                                                                         │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1d9c5885-3e56-4c57-be14-064b4e40da1a                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ```                                                                                              │
│  ===========================================                                                                    │
│  NON-LINEAR PROBABILITY MODEL                                                                                   │
│  ===========================================                                                                    │
│                                                                                                                 │
│  Data Exploration Summary:                                                                                      │
│  - The scoring function formula "mve * bm + bm * mom12m + mom12m * mom12m" was tested on S&P 500 firms data     │
│  for December 2020, revealing significant differences in scores between the top and bottom deciles.             │
│                                                                                                                 │
│  PROBABILITY FUNCTION (Python/Pseudo-code):                                                                     │
│  def calculate_buy_probability(mve, bm, mom12m):                                                                │
│      # Non-linear scoring function                                                                              │
│      score = mve * bm + bm * mom12m + mom12m * mom12m                                                           │
│      return score                                                                                               │
│                                                                                                                 │
│  Rationale:                                                                                                     │
│  - The scoring function incorporates interactions between firm size (mve), book-to-market ratio (bm), and       │
│  12-month momentum (mom12m).                                                                                    │
│  - The term mve * bm captures the combined effect of firm size and value factor, indicating potentially         │
│  undervalued larger firms.                                                                                      │
│  - The term bm * mom12m considers the interaction between value and momentum, suggesting momentum effects on    │
│  undervalued stocks.                                                                                            │
│  - The term mom12m * mom12m introduces non-linearity by emphasizing the momentum effect squared, reflecting     │
│  potential momentum compounding.                                                                                │
│                                                                                                                 │
│  FINAL RULE: The trading desk will buy the top decile and sell the bottom decile of S&P 500 stocks based on     │
│  the calculated buy probability scores from the non-linear scoring function.                                    │
│  ===========================================                                                                    │
│  ```                                                  


SYSTEMATIC STRATEGY OUTPUT
```
NON-LINEAR PROBABILITY MODEL

Data Exploration Summary:
- The scoring function formula "mve * bm + bm * mom12m + mom12m * mom12m" was tested on S&P 500 firms data for December 2020, revealing significant differences in scores between the top and bottom deciles.

PROBABILITY FUNCTION (Python/Pseudo-code):
def calculate_buy_probability(mve, bm, mom12m):
    # Non-linear scoring function
    score = mve * bm + bm * mom12m + mom12m * mom12m
    return score

Rationale:
- The scoring function incorporates interactions between firm size (mve), book-to-market ratio (bm), and 12-month momentum (mom12m). 
- The term mve * bm captures the combined effect of firm size and value factor, indicating potentially undervalued larger firms.
- The term bm * mom12m considers the interaction between value and momentum, suggesting momentum effects on undervalued stocks.
- The term mom12m * mom12m introduces non-linearity by emphasizing the momentum effect squared, reflecting 